# HungerHub Full Database Extraction - Capacity Analysis

## Objective
Test server capacity and database sizes to determine feasibility of downloading all data from Oracle databases.

## Tests to Perform:
1. **Server Resource Analysis**: RAM, disk space, CPU capacity
2. **Database Size Assessment**: Total records, data volume estimation
3. **Network Performance**: Data transfer speeds and timeouts
4. **Memory Stress Testing**: Progressively larger data loads
5. **Storage Capacity**: Available disk space for full extracts

## Risk Assessment:
- Memory exhaustion and notebook crashes
- Network timeouts and connection failures
- Disk space limitations
- Oracle connection pool exhaustion

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import psutil
import os
import sys
import time
from datetime import datetime
from pathlib import Path

print("Server Capacity Analysis Started")
print("=" * 50)
print(f"Analysis Time: {datetime.now()}")
print(f"Python Version: {sys.version}")

Server Capacity Analysis Started
Analysis Time: 2025-08-08 14:05:25.390295
Python Version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:09:02) [GCC 11.2.0]


## 1. Current Server Resource Analysis

In [2]:
# Comprehensive server resource analysis
def analyze_server_resources():
    """Analyze current server capacity"""
    
    print("CURRENT SERVER RESOURCES")
    print("=" * 40)
    
    # Memory analysis
    memory = psutil.virtual_memory()
    swap = psutil.swap_memory()
    
    print(f"\nMEMORY STATUS:")
    print(f"  Total RAM: {memory.total / (1024**3):.2f} GB")
    print(f"  Available RAM: {memory.available / (1024**3):.2f} GB")
    print(f"  Used RAM: {memory.used / (1024**3):.2f} GB ({memory.percent:.1f}%)")
    print(f"  Free RAM: {memory.free / (1024**3):.2f} GB")
    print(f"  Cached: {memory.cached / (1024**3):.2f} GB")
    print(f"  Buffers: {memory.buffers / (1024**3):.2f} GB")
    
    print(f"\nSWAP STATUS:")
    print(f"  Total Swap: {swap.total / (1024**3):.2f} GB")
    print(f"  Used Swap: {swap.used / (1024**3):.2f} GB ({swap.percent:.1f}%)")
    print(f"  Free Swap: {swap.free / (1024**3):.2f} GB")
    
    # CPU analysis
    cpu_count = psutil.cpu_count()
    cpu_usage = psutil.cpu_percent(interval=1)
    load_avg = os.getloadavg()
    
    print(f"\nCPU STATUS:")
    print(f"  CPU Cores: {cpu_count}")
    print(f"  CPU Usage: {cpu_usage:.1f}%")
    print(f"  Load Average: {load_avg[0]:.2f}, {load_avg[1]:.2f}, {load_avg[2]:.2f}")
    
    # Disk analysis for key mount points
    print(f"\nDISK STATUS:")
    key_paths = ['/', '/home', '/tmp']
    
    for path in key_paths:
        if os.path.exists(path):
            disk = psutil.disk_usage(path)
            print(f"  {path}: {disk.total / (1024**3):.1f} GB total, "
                  f"{disk.free / (1024**3):.1f} GB free ({(disk.free/disk.total)*100:.1f}% free)")
    
    # Current working directory space
    cwd_disk = psutil.disk_usage('.')
    print(f"  Current dir: {cwd_disk.free / (1024**3):.1f} GB available")
    
    # Process information
    process = psutil.Process()
    proc_memory = process.memory_info()
    
    print(f"\nCURRENT PROCESS:")
    print(f"  Process RSS: {proc_memory.rss / (1024**2):.1f} MB")
    print(f"  Process VMS: {proc_memory.vms / (1024**2):.1f} MB")
    print(f"  Process CPU: {process.cpu_percent():.1f}%")
    
    return {
        'total_ram_gb': memory.total / (1024**3),
        'available_ram_gb': memory.available / (1024**3),
        'free_disk_gb': cwd_disk.free / (1024**3),
        'cpu_cores': cpu_count,
        'current_memory_mb': proc_memory.rss / (1024**2)
    }

server_resources = analyze_server_resources()

CURRENT SERVER RESOURCES

MEMORY STATUS:
  Total RAM: 15.57 GB
  Available RAM: 10.14 GB
  Used RAM: 5.09 GB (34.9%)
  Free RAM: 0.43 GB
  Cached: 9.66 GB
  Buffers: 0.39 GB

SWAP STATUS:
  Total Swap: 0.00 GB
  Used Swap: 0.00 GB (0.0%)
  Free Swap: 0.00 GB

CPU STATUS:
  CPU Cores: 4
  CPU Usage: 5.3%
  Load Average: 0.08, 0.04, 0.06

DISK STATUS:
  /: 246.9 GB total, 232.7 GB free (94.2% free)
  /home: 246.9 GB total, 232.7 GB free (94.2% free)
  /tmp: 246.9 GB total, 232.7 GB free (94.2% free)
  Current dir: 232.7 GB available

CURRENT PROCESS:
  Process RSS: 169.1 MB
  Process VMS: 1137.7 MB
  Process CPU: 0.0%


## 2. Database Size Assessment

In [3]:
# Load existing Oracle connection capability
import sys
import os
from pathlib import Path

# Add the main notebook's path to access Oracle connector
sys.path.append('..')
sys.path.append('../src')

# Try to load Oracle libraries and connection info
try:
    import cx_Oracle
    from dotenv import load_dotenv
    
    # Load environment variables
    env_file = Path('../config/.env')
    if env_file.exists():
        load_dotenv(env_file)
        print("✅ Oracle libraries and config loaded")
    else:
        print("❌ Config file not found")
        
except ImportError as e:
    print(f"❌ Oracle libraries not available: {e}")
    print("Cannot perform database size assessment without Oracle connectivity")

✅ Oracle libraries and config loaded


In [4]:
# Database size estimation function
def estimate_database_sizes():
    """Estimate total database sizes and feasibility of full extraction"""
    
    databases = {
        'choice': {
            'host': os.getenv('CHOICE_ORACLE_HOST'),
            'port': os.getenv('CHOICE_ORACLE_PORT', '1521'),
            'service': os.getenv('CHOICE_ORACLE_SERVICE_NAME'),
            'user': os.getenv('CHOICE_USERNAME'),
            'password': os.getenv('CHOICE_PASSWORD'),
            'name': 'Choice Sandbox'
        },
        'agency': {
            'host': os.getenv('AGENCY_ORACLE_HOST'),
            'port': os.getenv('AGENCY_ORACLE_PORT', '1521'),
            'service': os.getenv('AGENCY_ORACLE_SERVICE_NAME'),
            'user': os.getenv('AGENCY_USERNAME'),
            'password': os.getenv('AGENCY_PASSWORD'),
            'name': 'Agency Sandbox'
        }
    }
    
    print("DATABASE SIZE ESTIMATION")
    print("=" * 40)
    
    total_estimated_records = 0
    total_estimated_size_gb = 0
    database_info = {}
    
    for db_key, config in databases.items():
        print(f"\nAnalyzing {config['name']}...")
        
        # Check if configuration is available
        missing_config = [k for k, v in config.items() if not v]
        if missing_config:
            print(f"❌ Missing configuration: {', '.join(missing_config)}")
            continue
        
        try:
            # Connect to database
            dsn = cx_Oracle.makedsn(config['host'], config['port'], service_name=config['service'])
            connection = cx_Oracle.connect(config['user'], config['password'], dsn)
            cursor = connection.cursor()
            
            # Get comprehensive table statistics
            size_query = """
            SELECT 
                table_name,
                num_rows,
                blocks,
                empty_blocks,
                avg_row_len,
                sample_size,
                last_analyzed,
                (num_rows * avg_row_len) as estimated_bytes
            FROM user_tables 
            WHERE num_rows > 0
            ORDER BY num_rows DESC
            """
            
            cursor.execute(size_query)
            results = cursor.fetchall()
            
            db_total_records = 0
            db_total_bytes = 0
            table_count = len(results)
            
            print(f"Found {table_count} tables with data")
            print("\nTop 10 Largest Tables:")
            print("-" * 70)
            print(f"{'Table Name':<25} {'Records':<12} {'Avg Row':<10} {'Est. Size':<12}")
            print("-" * 70)
            
            for i, row in enumerate(results[:10]):
                table_name, num_rows, blocks, empty_blocks, avg_row_len, sample_size, last_analyzed, est_bytes = row
                
                if num_rows and avg_row_len:
                    size_mb = (num_rows * avg_row_len) / (1024 * 1024)
                    print(f"{table_name:<25} {num_rows:<12,} {avg_row_len:<10} {size_mb:<12.1f} MB")
                    
                    db_total_records += num_rows
                    db_total_bytes += (num_rows * avg_row_len)
            
            # Calculate totals for all tables
            for row in results:
                num_rows, avg_row_len = row[1], row[4]
                if num_rows and avg_row_len:
                    if row not in results[:10]:  # Don't double count top 10
                        db_total_records += num_rows
                        db_total_bytes += (num_rows * avg_row_len)
            
            db_size_gb = db_total_bytes / (1024**3)
            
            print(f"\nDatabase Totals:")
            print(f"  Total Records: {db_total_records:,}")
            print(f"  Estimated Size: {db_size_gb:.2f} GB")
            print(f"  Average Row Size: {db_total_bytes/db_total_records:.0f} bytes" if db_total_records > 0 else "")
            
            total_estimated_records += db_total_records
            total_estimated_size_gb += db_size_gb
            
            database_info[db_key] = {
                'name': config['name'],
                'tables': table_count,
                'records': db_total_records,
                'size_gb': db_size_gb,
                'largest_tables': results[:5]  # Store top 5 for later reference
            }
            
            cursor.close()
            connection.close()
            
        except Exception as e:
            print(f"❌ Error analyzing {config['name']}: {e}")
            continue
    
    print(f"\n" + "="*50)
    print(f"TOTAL DATABASE SIZE ESTIMATE")
    print(f"="*50)
    print(f"Total Records: {total_estimated_records:,}")
    print(f"Total Size: {total_estimated_size_gb:.2f} GB")
    print(f"Databases: {len(database_info)}")
    
    return {
        'total_records': total_estimated_records,
        'total_size_gb': total_estimated_size_gb,
        'databases': database_info
    }

# Run database size estimation if Oracle is available
try:
    if 'cx_Oracle' in globals():
        database_sizes = estimate_database_sizes()
    else:
        print("❌ Oracle not available for database size estimation")
        database_sizes = None
except Exception as e:
    print(f"❌ Database size estimation failed: {e}")
    database_sizes = None

DATABASE SIZE ESTIMATION

Analyzing Choice Sandbox...
Found 178 tables with data

Top 10 Largest Tables:
----------------------------------------------------------------------
Table Name                Records      Avg Row    Est. Size   
----------------------------------------------------------------------
RW_FLEX                   4,393,524    49         205.3        MB
RW_PO_FLEX                578,995      54         29.8         MB
RW_CONTACT_INFO           566,783      157        84.9         MB
ACSHARES_ARCHIVE          554,550      100        52.9         MB
RW_SHPMNT_METHOD          231,675      35         7.7          MB
RW_ORDR_ITM_SCHDL         230,282      132        29.0         MB
RW_ORDER_ITEM             230,282      236        51.8         MB
RW_DOC_TRANS              108,861      55         5.7          MB
BIDDINGSESSION_SNAPSHOT   102,470      18         1.8          MB
RW_ORDER_SUPPLIER         96,552       57         5.2          MB

Database Totals:
  Total Reco

## 3. Feasibility Analysis & Recommendations

In [5]:
# Comprehensive feasibility analysis
def analyze_full_extraction_feasibility(server_resources, database_sizes):
    """Analyze feasibility of full database extraction"""
    
    print("FULL EXTRACTION FEASIBILITY ANALYSIS")
    print("=" * 50)
    
    if not database_sizes:
        print("❌ Cannot analyze feasibility without database size information")
        return None
    
    # Resource requirements estimation
    data_size_gb = database_sizes['total_size_gb']
    total_records = database_sizes['total_records']
    
    # Memory requirements (assuming pandas DataFrame overhead)
    # Rule of thumb: DataFrames use 3-5x raw data size in memory
    memory_multiplier = 4  # Conservative estimate
    required_memory_gb = data_size_gb * memory_multiplier
    
    # Storage requirements (raw + processed + backups)
    # CSV files are typically larger than raw data
    # Parquet files are typically smaller
    storage_multiplier = 3  # CSV + Parquet + working space
    required_storage_gb = data_size_gb * storage_multiplier
    
    print(f"RESOURCE REQUIREMENTS ESTIMATION:")
    print(f"  Raw Data Size: {data_size_gb:.2f} GB")
    print(f"  Required Memory: {required_memory_gb:.2f} GB (includes DataFrame overhead)")
    print(f"  Required Storage: {required_storage_gb:.2f} GB (includes processing files)")
    print(f"  Total Records: {total_records:,}")
    
    print(f"\nAVAILABLE RESOURCES:")
    print(f"  Available Memory: {server_resources['available_ram_gb']:.2f} GB")
    print(f"  Available Storage: {server_resources['free_disk_gb']:.2f} GB")
    print(f"  CPU Cores: {server_resources['cpu_cores']}")
    
    # Feasibility assessment
    memory_feasible = server_resources['available_ram_gb'] >= required_memory_gb
    storage_feasible = server_resources['free_disk_gb'] >= required_storage_gb
    
    print(f"\nFEASIBILITY ASSESSMENT:")
    memory_status = "✅ FEASIBLE" if memory_feasible else "❌ INSUFFICIENT"
    storage_status = "✅ FEASIBLE" if storage_feasible else "❌ INSUFFICIENT"
    
    print(f"  Memory: {memory_status}")
    if not memory_feasible:
        shortage = required_memory_gb - server_resources['available_ram_gb']
        print(f"    Need additional {shortage:.2f} GB memory")
    
    print(f"  Storage: {storage_status}")
    if not storage_feasible:
        shortage = required_storage_gb - server_resources['free_disk_gb']
        print(f"    Need additional {shortage:.2f} GB storage")
    
    # Overall feasibility
    overall_feasible = memory_feasible and storage_feasible
    
    print(f"\nOVERALL FEASIBILITY: {'✅ POSSIBLE' if overall_feasible else '❌ NOT RECOMMENDED'}")
    
    # Alternative approaches
    print(f"\nRECOMMENDATIONS:")
    
    if overall_feasible:
        print("  ✅ Full extraction appears feasible with current resources")
        print("  ✅ Consider implementing chunked processing for safety")
        print("  ✅ Monitor memory usage during extraction")
        print("  ✅ Implement progress checkpoints for recovery")
    else:
        print("  🔄 ALTERNATIVE APPROACHES:")
        
        if not memory_feasible:
            print("    - Implement streaming/chunked processing (process tables one at a time)")
            print("    - Use database-side aggregation to reduce data volume")
            print("    - Consider upgrading server memory")
            
        if not storage_feasible:
            print("    - Use compression (gzip CSV, optimized Parquet)")
            print("    - Implement direct database-to-analysis without full extraction")
            print("    - Consider cloud storage for processed files")
        
        print("    - Selective extraction (prioritize most important tables)")
        print("    - Implement table-by-table processing pipeline")
    
    return {
        'feasible': overall_feasible,
        'required_memory_gb': required_memory_gb,
        'required_storage_gb': required_storage_gb,
        'memory_feasible': memory_feasible,
        'storage_feasible': storage_feasible
    }

# Run feasibility analysis
if 'server_resources' in globals() and 'database_sizes' in globals():
    feasibility = analyze_full_extraction_feasibility(server_resources, database_sizes)
else:
    print("❌ Cannot perform feasibility analysis - missing resource or database information")
    feasibility = None

FULL EXTRACTION FEASIBILITY ANALYSIS
RESOURCE REQUIREMENTS ESTIMATION:
  Raw Data Size: 1.64 GB
  Required Memory: 6.57 GB (includes DataFrame overhead)
  Required Storage: 4.93 GB (includes processing files)
  Total Records: 18,967,378

AVAILABLE RESOURCES:
  Available Memory: 10.14 GB
  Available Storage: 232.73 GB
  CPU Cores: 4

FEASIBILITY ASSESSMENT:
  Memory: ✅ FEASIBLE
  Storage: ✅ FEASIBLE

OVERALL FEASIBILITY: ✅ POSSIBLE

RECOMMENDATIONS:
  ✅ Full extraction appears feasible with current resources
  ✅ Consider implementing chunked processing for safety
  ✅ Monitor memory usage during extraction
  ✅ Implement progress checkpoints for recovery


## 4. Implementation Strategy

Based on the analysis results, here are the recommended implementation strategies for full data extraction.

In [6]:
# Implementation strategy recommendations
def generate_implementation_strategy(feasibility_results, database_info):
    """Generate specific implementation strategy based on analysis"""
    
    print("IMPLEMENTATION STRATEGY")
    print("=" * 40)
    
    if not feasibility_results:
        print("❌ Cannot generate strategy without feasibility analysis")
        return
    
    if feasibility_results['feasible']:
        print("📋 FULL EXTRACTION STRATEGY:")
        print("\n1. PREPARATION:")
        print("   - Create dedicated output directories with sufficient space")
        print("   - Implement memory monitoring and alerts")
        print("   - Set up progress checkpoints and recovery mechanisms")
        
        print("\n2. EXTRACTION APPROACH:")
        print("   - Process databases sequentially (not parallel) to manage memory")
        print("   - Use chunked reading for very large tables (>1M records)")
        print("   - Implement immediate compression (Parquet format)")
        print("   - Clear memory between tables (garbage collection)")
        
        print("\n3. OPTIMIZATION:")
        print("   - Use Oracle ROWID-based pagination for large tables")
        print("   - Implement parallel processing within tables (column-wise)")
        print("   - Use database-side filtering where possible")
        
    else:
        print("📋 CONSTRAINED EXTRACTION STRATEGY:")
        print("\n1. CHUNKED PROCESSING APPROACH:")
        print("   - Process one table at a time, clear memory after each")
        print("   - Use streaming approach: Database → Process → Save → Clear")
        print("   - Implement table priority ranking (start with most critical)")
        
        print("\n2. MEMORY MANAGEMENT:")
        print("   - Use generators and iterators instead of loading full DataFrames")
        print("   - Implement chunk size based on available memory")
        print("   - Use database cursors with arraysize optimization")
        
        print("\n3. STORAGE OPTIMIZATION:")
        print("   - Use compressed formats (gzip, bz2) for immediate compression")
        print("   - Implement data deduplication at extraction time")
        print("   - Consider external storage mounting if available")
    
    # Generate sample code template
    print("\n" + "="*50)
    print("SAMPLE IMPLEMENTATION CODE TEMPLATE:")
    print("="*50)
    
    code_template = '''
def extract_full_database(db_config, output_dir, chunk_size=10000):
    """Extract full database with memory management"""
    
    import gc
    import psutil
    
    connection = establish_connection(db_config)
    tables = get_table_list(connection)
    
    for table_info in tables:
        table_name = table_info['name']
        row_count = table_info['rows']
        
        print(f"Extracting {table_name} ({row_count:,} rows)...")
        
        if row_count > chunk_size:
            # Chunked processing for large tables
            extract_table_chunked(connection, table_name, output_dir, chunk_size)
        else:
            # Direct extraction for small tables
            extract_table_direct(connection, table_name, output_dir)
        
        # Memory cleanup after each table
        gc.collect()
        
        # Memory usage check
        memory_percent = psutil.virtual_memory().percent
        if memory_percent > 80:
            print(f"Warning: High memory usage ({memory_percent:.1f}%)")
            time.sleep(2)  # Brief pause for system recovery
    
    connection.close()
    print("Full database extraction completed!")
    '''
    
    print(code_template)
    
    # Save analysis results
    results_summary = {
        'analysis_date': datetime.now().isoformat(),
        'feasibility': feasibility_results,
        'server_resources': server_resources,
        'database_sizes': database_sizes if 'database_sizes' in globals() else None,
        'recommendation': 'full_extraction' if feasibility_results['feasible'] else 'chunked_processing'
    }
    
    # Save to file
    import json
    with open('capacity_analysis_results.json', 'w') as f:
        json.dump(results_summary, f, indent=2, default=str)
    
    print(f"\n✅ Analysis results saved to: capacity_analysis_results.json")
    
    return results_summary

# Generate implementation strategy
if 'feasibility' in globals() and feasibility:
    strategy = generate_implementation_strategy(feasibility, database_sizes if 'database_sizes' in globals() else None)
else:
    print("❌ Cannot generate implementation strategy without feasibility analysis")

IMPLEMENTATION STRATEGY
📋 FULL EXTRACTION STRATEGY:

1. PREPARATION:
   - Create dedicated output directories with sufficient space
   - Implement memory monitoring and alerts
   - Set up progress checkpoints and recovery mechanisms

2. EXTRACTION APPROACH:
   - Process databases sequentially (not parallel) to manage memory
   - Use chunked reading for very large tables (>1M records)
   - Implement immediate compression (Parquet format)
   - Clear memory between tables (garbage collection)

3. OPTIMIZATION:
   - Use Oracle ROWID-based pagination for large tables
   - Implement parallel processing within tables (column-wise)
   - Use database-side filtering where possible

SAMPLE IMPLEMENTATION CODE TEMPLATE:

def extract_full_database(db_config, output_dir, chunk_size=10000):
    """Extract full database with memory management"""
    
    import gc
    import psutil
    
    connection = establish_connection(db_config)
    tables = get_table_list(connection)
    
    for table_info in 